## Operator Study

This notebook focuses on studying single operator surgery, comparing different approaches against baseline. We also test different compression strategies and the effects of combining them (operator replacement, low-rank factorization, ...)

### Operator architecture

For the current dense-model scope, write the teacher SwiGLU MLP at layer $l$ as

$$
f_l(h)=W_{\mathrm{down}}\left[\mathrm{SiLU}(W_{\mathrm{gate}}h)\odot(W_{\mathrm{up}}h)\right],
$$

$h$ is the normalized MLP input and $f_l(h)$ is the contribution returned to the residual stream. A drop-in operator $\hat f_l$ replaces only $f_l$, with the same input and output shape; the surrounding Transformer block remains unchanged.

### Operator classes: Design space relevant to this thesis

The table maps candidate operators to the question each one can answer. Parameter scales ignore biases, with $d=d_{\mathrm{model}}$ and reduced width or rank $r<d$.

| Operator class | Representative form | Approximate parameters | Experimental role |
| --- | --- | ---: | --- |
| Zero / mean controls | $0$ or $\mu_y$ | 0 trainable | Bound complete removal and input-independent prediction. |
| Dense affine map | $Ax+b$ | $d^2+d$ | Test whether one learned affine transformation is sufficient. |
| Low-rank affine map | $U(Vx)+b$ | $2dr+d$ | Test whether linear structure is sufficient under a rank bottleneck. |
| Compact ungated MLP | $W_2\phi(W_1x)$ | $2dr$ | Isolate the value of nonlinearity without multiplicative gating. |
| Reduced-width SwiGLU | $W_d[\mathrm{SiLU}(W_gx)\odot(W_ux)]$ | $3dr$ | Preserve the teacher family at lower width; this is the current practical baseline. |
| Linear plus nonlinear correction | $U(Vx)+W_2\phi(W_1x)$ | $2d(r_{\mathrm{lin}}+r_{\mathrm{nl}})$ | Test whether most behavior is low-rank linear with a small nonlinear residual. |
| Factorized teacher projections | $W_j\approx U_jV_j$ inside SwiGLU | depends on selected matrices and ranks | Compress original weights while retaining the gate and activation structure. |
| Partial internal replacement | Replace only a projection, gate branch, or activation path | design-specific | Test whether targeted surgery outperforms replacing the complete MLP. |

Whole-block operators are studied first because they share one clean drop-in interface. Partial replacement and factorization are later experiments and require their own controls.

**Status and citation requirement.** This table is a project design-space synthesis, not a taxonomy copied from one paper. The affine, low-rank, compact-MLP, and hybrid equations are explanatory operator definitions and do not require citations as proposed experimental variants. Any claim that a particular published method uses or benefits from one of them must cite that method.

### Low-rank factorization: two distinct uses

1. **Whole-operator approximation:** $\hat f(x)=U(Vx)+b$ replaces the nonlinear MLP with one low-rank affine function.
2. **Internal weight factorization:** $W_j\approx U_jV_j$ replaces selected SwiGLU weight matrices while retaining the nonlinear gated computation.

These approaches may use the same matrix factorization machinery, but they test different hypotheses and should be reported separately.

### Setup

### Experiments

#### Operator classes